# Python Data Analysis Playground

Start with the wide retail table, then load individual tables as needed. All data is synthetic.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("data")
transactions = pd.read_csv(DATA / "retail_transactions.csv", parse_dates=["order_date", "ship_date", "delivery_date", "signup_date"])
customers = pd.read_csv(DATA / "customers.csv", parse_dates=["signup_date"])
products = pd.read_csv(DATA / "products.csv", parse_dates=["launch_date"])
orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date", "ship_date", "delivery_date", "promised_delivery_date"])
items = pd.read_csv(DATA / "order_items.csv")
daily = pd.read_csv(DATA / "daily_regional_metrics.csv", parse_dates=["date"])
events = pd.read_csv(DATA / "website_events.csv", parse_dates=["event_timestamp"])

print("Transactions shape:", transactions.shape)
transactions.head()

## First exploration
Run each cell, then try changing the columns or conditions.

In [ ]:
transactions.info()
transactions.describe(include="all").T

In [ ]:
transactions.isna().sum().sort_values(ascending=False).head(15)

## NumPy practice: vectorized calculations
Create arrays from columns and calculate a simple margin.

In [ ]:
prices = products["list_price"].to_numpy()
costs = products["unit_cost"].to_numpy()
margin = prices - costs
margin_pct = np.divide(margin, prices, out=np.zeros_like(margin, dtype=float), where=prices != 0)
print("Median margin:", np.median(margin).round(2))
print("90th percentile list price:", np.quantile(prices, 0.90).round(2))
print("Highest-margin product row index:", np.argmax(margin))

## pandas practice: filtering, grouping and sorting

In [ ]:
high_value = transactions.loc[transactions["net_sales"] >= 5000, ["order_id", "product_name", "category", "quantity", "net_sales"]]
high_value.head(10)

In [ ]:
transactions.groupby("category", dropna=False).agg(
    line_count=("order_item_id", "count"),
    units_sold=("quantity", "sum"),
    net_sales=("net_sales", "sum"),
    average_line_sales=("net_sales", "mean")
).sort_values("net_sales", ascending=False)

## Joins: understand relational data

In [ ]:
orders_with_customer = orders.merge(customers, on="customer_id", how="left", validate="many_to_one", indicator=True)
orders_with_customer["_merge"].value_counts()

## Time series: rolling average

In [ ]:
north = (daily.loc[daily["region"] == "North"]
         .sort_values("date")
         .set_index("date"))
north["revenue_7_day_average"] = north["revenue"].rolling("7D").mean()
north[["revenue", "revenue_7_day_average"]].head(12)

## Cleaning challenge
Use `messy_sales.csv` to standardize city labels, parse dates, convert sales to numeric values, identify duplicates and negative/outlier values. Record your cleaning decisions before removing any row.

In [ ]:
messy = pd.read_csv(DATA / "messy_sales.csv")
print("Shape:", messy.shape)
print("Duplicate rows:", messy.duplicated().sum())
messy.head(10)

For many more tasks, open `PRACTICE_ROADMAP.md`. It is a broad reference map from NumPy/pandas concepts to relevant files and exercises.